# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MbaliThusi/Mbali-Flyrank-intern-rep./blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My provisional lane is Lane 4: CTR / Engagement Opportunity Scoring. I chose it because it ends in a clear action: a ranked list of visible pages that get fewer clicks than they should, so a content team knows which titles and meta descriptions to review first. The data I need (impressions, clicks, CTR and position) is in the starter dataset, and the method is simple enough to explain: compare each page to the expected CTR for its position tier. I can still change lanes until the end of Week 4.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Question:** Which visible pages get fewer clicks than expected for their position, and deserve a metadata, content or monitoring review?

**Unit of analysis:** one page.

**Output:** a ranked list of review candidates, each with reason codes (for example high impressions, low CTR, enough volume) and a suggested action (rewrite title/meta, improve intent match, or monitor).

**Decision and action:** A content team decides which pages to review first. Someone then rewrites the title or meta description, improves how well the page matches search intent, or keeps watching it.

**Cost of a wrong call:** Editor time is wasted on pages that were fine, and real opportunities are missed. A low CTR does not always mean a bad title, so the list is a place to look first, not a verdict.

**Why data or ML can help:** There are too many pages to check by hand, and comparing CTR across different positions is misleading. A simple expected-CTR baseline per position tier is a fairer starting point. This is not just "train a model": the first step is a transparent comparison that anyone can check.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
from google.colab import files
import pandas as pd

uploaded = files.upload()          # a "Choose files" button appears below the cell
fname = list(uploaded.keys())[0]
print("Loaded:", fname)

df = pd.read_csv(fname)

print("Rows, columns:", df.shape)
print("Columns:", df.columns.tolist())

def find(*keys):
    for c in df.columns:
        if any(k in c.lower() for k in keys):
            return c

imp, clk, ctr, pos = find("impression"), find("click"), find("ctr"), find("position")
print("Detected:", {"impressions": imp, "clicks": clk, "ctr": ctr, "position": pos})

if imp and clk:
    print("Pages:", len(df))
    print("Total impressions:", int(df[imp].sum()))
    print("Overall CTR: %.2f%%" % (df[clk].sum() / df[imp].sum() * 100))
if pos and (ctr or (imp and clk)):
    if not ctr:
        df["ctr_calc"] = df[clk] / df[imp]
        ctr = "ctr_calc"
    df["position_tier"] = pd.cut(df[pos], [0, 3, 10, 20, 10000], labels=["1-3", "4-10", "11-20", "20+"])
    print(df.groupby("position_tier", observed=True)[ctr].agg(["count", "mean"]))

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv
Loaded: content_refresh_anonymized (1).csv
Rows, columns: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Detected: {'impressions': 'impressions_90d', 'clicks': 'clicks_90d', 'ctr'

**What the starter data shows:**
- The dataset has 30,000 pages and about 156 million impressions over 90 days, so there is plenty of volume to work with.
- Overall CTR is 0.31%, which means most impressions do not turn into clicks.
- Average CTR falls steeply by position tier: about 2.71 for positions 1-3, 0.65 for positions 4-10, 0.32 for positions 11-20 and 0.21 for positions 20+. Pages in the top tier get roughly 4 times the CTR of the next tier.

**Why this makes Lane 4 worth the next 7 weeks:** Position clearly shapes CTR, so a fair review list must compare each page only to other pages in its own position tier. A page with a low CTR for its tier and high impressions is a sensible candidate for a title or meta review. The numbers suggest there is a real pattern to build a simple expected-CTR baseline on.

**Notes on these numbers:** They are observational and describe the starter data only. About 1,200 pages are not in any tier (probably missing position values), so I will check those before ranking. The CTR column looks like it is stored as a percentage, which I will confirm in the data dictionary.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**What this work can say:**
- It is observational: I looked at existing data and did not run an experiment.
- It is directional: pages with a CTR well below the average for their position tier are worth a closer look.
- It is decision support: it gives a content team a ranked list of where to look first.

**What it cannot say:**
- It cannot prove that a bad title or meta description is the cause of a low CTR. Other reasons are possible, such as search intent, SERP features, or seasonality.
- It cannot prove that rewriting a page will raise clicks.
- It does not predict or explain Google's algorithm.
- Pages with few impressions are noisy, so I will set a minimum-volume threshold before ranking anything.
- Position is an average, so comparisons are only fair within the same position tier.

I will use words like "suggests", "is associated with", and "worth reviewing" instead of "proves" or "causes".

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.